# Tag Standardization Methods Comparison

This notebook compares different methods for semantically standardizing tags to reduce redundancy.

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
import warnings
warnings.filterwarnings('ignore')

## Load Tags from Cache Files

In [ ]:
# Load image tags (brain tumor)
with open('../input/brain_tumor_images/tags_cache_old.json', 'r') as f:
    image_tags_raw = json.load(f)

# Load text tags (SMS spam)
with open('../input/sms_spam/tags_cache_old.json', 'r') as f:
    text_tags_raw = json.load(f)

print(f"Image samples: {len(image_tags_raw)}")
print(f"Text samples: {len(text_tags_raw)}")

In [ ]:
def extract_valid_tags(tags_dict: dict) -> list[str]:
    """Extract all unique tags, filtering out error messages."""
    all_tags = []
    for tags in tags_dict.values():
        for tag in tags:
            # Filter out error messages (they contain spaces and are long)
            if len(tag) < 50 and ' ' not in tag and not tag.startswith("I'm") and not tag.startswith("I can"):
                all_tags.append(tag.lower())
    return list(set(all_tags))

image_tags = extract_valid_tags(image_tags_raw)
text_tags = extract_valid_tags(text_tags_raw)

print(f"Unique image tags: {len(image_tags)}")
print(f"Unique text tags: {len(text_tags)}")

In [ ]:
# Preview some tags
print("Sample image tags:")
print(sorted(image_tags)[:20])
print("\nSample text tags:")
print(sorted(text_tags)[:20])

## Method 1: TF-IDF with Character N-grams (Zero Dependencies)

In [ ]:
def standardize_tfidf(tags: list[str], distance_threshold: float = 0.5) -> dict[str, str]:
    """Standardize tags using TF-IDF character n-grams."""
    if len(tags) < 2:
        return {t: t for t in tags}
    
    # Preprocess: replace underscores with spaces
    processed = [t.replace('_', ' ') for t in tags]
    
    # Character n-grams capture morphological similarity
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4))
    embeddings = vectorizer.fit_transform(processed)
    
    # Compute distance matrix
    distances = cosine_distances(embeddings)
    
    # Cluster
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='precomputed',
        linkage='average'
    )
    labels = clustering.fit_predict(distances)
    
    # Map to canonical (shortest tag in cluster)
    mapping = {}
    for cluster_id in set(labels):
        cluster_tags = [t for t, l in zip(tags, labels) if l == cluster_id]
        canonical = min(cluster_tags, key=len)
        for tag in cluster_tags:
            mapping[tag] = canonical
    
    return mapping

## Method 2: Sentence Transformers (MiniLM)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the lightweight model
st_model = SentenceTransformer('all-MiniLM-L6-v2')

def standardize_sentence_transformers(tags: list[str], distance_threshold: float = 0.4) -> dict[str, str]:
    """Standardize tags using sentence-transformers embeddings."""
    if len(tags) < 2:
        return {t: t for t in tags}
    
    # Preprocess: replace underscores with spaces
    processed = [t.replace('_', ' ') for t in tags]
    
    # Get embeddings
    embeddings = st_model.encode(processed)
    
    # Cluster
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)
    
    # Map to canonical (shortest tag in cluster)
    mapping = {}
    for cluster_id in set(labels):
        cluster_tags = [t for t, l in zip(tags, labels) if l == cluster_id]
        canonical = min(cluster_tags, key=len)
        for tag in cluster_tags:
            mapping[tag] = canonical
    
    return mapping

## Method 3: spaCy Word Vectors

In [ ]:
import spacy

# Try to load medium model with vectors, fall back to small
try:
    nlp = spacy.load('en_core_web_md')
    print("Loaded en_core_web_md")
except OSError:
    print("en_core_web_md not found, trying en_core_web_sm...")
    try:
        nlp = spacy.load('en_core_web_sm')
        print("Loaded en_core_web_sm (note: limited vectors)")
    except OSError:
        nlp = None
        print("No spaCy model available, skipping spaCy method")

def standardize_spacy(tags: list[str], distance_threshold: float = 0.5) -> dict[str, str]:
    """Standardize tags using spaCy word vectors."""
    if nlp is None or len(tags) < 2:
        return {t: t for t in tags}
    
    # Get embeddings
    processed = [t.replace('_', ' ') for t in tags]
    embeddings = np.array([nlp(text).vector for text in processed])
    
    # Check if vectors are all zeros (no word vectors in model)
    if np.all(embeddings == 0):
        print("Warning: spaCy model has no word vectors")
        return {t: t for t in tags}
    
    # Cluster
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)
    
    # Map to canonical
    mapping = {}
    for cluster_id in set(labels):
        cluster_tags = [t for t, l in zip(tags, labels) if l == cluster_id]
        canonical = min(cluster_tags, key=len)
        for tag in cluster_tags:
            mapping[tag] = canonical
    
    return mapping

## Run Comparison

In [ ]:
def get_clusters(mapping: dict[str, str]) -> dict[str, list[str]]:
    """Convert mapping to clusters for display."""
    clusters = {}
    for original, canonical in mapping.items():
        if canonical not in clusters:
            clusters[canonical] = []
        if original != canonical:
            clusters[canonical].append(original)
    # Only return clusters with merged tags
    return {k: v for k, v in clusters.items() if v}

def compare_methods(tags: list[str], tag_type: str):
    """Compare all methods on a set of tags."""
    print(f"\n{'='*60}")
    print(f"Comparing methods for {tag_type.upper()} tags")
    print(f"{'='*60}")
    print(f"\nOriginal unique tags: {len(tags)}")
    
    results = {}
    
    # Method 1: TF-IDF
    print("\n" + "-"*40)
    print("Method 1: TF-IDF Character N-grams")
    print("-"*40)
    mapping_tfidf = standardize_tfidf(tags, distance_threshold=0.5)
    canonical_tfidf = set(mapping_tfidf.values())
    clusters_tfidf = get_clusters(mapping_tfidf)
    print(f"Final tags: {len(canonical_tfidf)} (reduced by {len(tags) - len(canonical_tfidf)})")
    results['TF-IDF'] = {'canonical': canonical_tfidf, 'clusters': clusters_tfidf, 'mapping': mapping_tfidf}
    
    # Method 2: Sentence Transformers
    print("\n" + "-"*40)
    print("Method 2: Sentence Transformers (MiniLM)")
    print("-"*40)
    mapping_st = standardize_sentence_transformers(tags, distance_threshold=0.4)
    canonical_st = set(mapping_st.values())
    clusters_st = get_clusters(mapping_st)
    print(f"Final tags: {len(canonical_st)} (reduced by {len(tags) - len(canonical_st)})")
    results['SentenceTransformers'] = {'canonical': canonical_st, 'clusters': clusters_st, 'mapping': mapping_st}
    
    # Method 3: spaCy
    print("\n" + "-"*40)
    print("Method 3: spaCy Word Vectors")
    print("-"*40)
    mapping_spacy = standardize_spacy(tags, distance_threshold=0.5)
    canonical_spacy = set(mapping_spacy.values())
    clusters_spacy = get_clusters(mapping_spacy)
    print(f"Final tags: {len(canonical_spacy)} (reduced by {len(tags) - len(canonical_spacy)})")
    results['spaCy'] = {'canonical': canonical_spacy, 'clusters': clusters_spacy, 'mapping': mapping_spacy}
    
    return results

In [ ]:
# Run comparison for image tags
image_results = compare_methods(image_tags, "image")

In [ ]:
# Run comparison for text tags
text_results = compare_methods(text_tags, "text")

## Summary Comparison

In [ ]:
# Create summary table
summary_data = []
for tag_type, results, original_count in [
    ('Image', image_results, len(image_tags)),
    ('Text', text_results, len(text_tags))
]:
    for method, data in results.items():
        summary_data.append({
            'Tag Type': tag_type,
            'Method': method,
            'Original Tags': original_count,
            'Final Tags': len(data['canonical']),
            'Reduction': original_count - len(data['canonical']),
            'Reduction %': f"{(original_count - len(data['canonical'])) / original_count * 100:.1f}%",
            'Clusters Merged': len(data['clusters'])
        })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*70)
print("SUMMARY COMPARISON")
print("="*70)
print(summary_df.to_string(index=False))

## Detailed Cluster Review

Review which tags were merged together by each method.

In [ ]:
def display_clusters(results: dict, tag_type: str):
    """Display merged clusters for review."""
    print(f"\n{'='*70}")
    print(f"MERGED CLUSTERS FOR {tag_type.upper()} TAGS")
    print(f"{'='*70}")
    
    for method, data in results.items():
        print(f"\n{'-'*50}")
        print(f"{method}")
        print(f"{'-'*50}")
        
        clusters = data['clusters']
        if not clusters:
            print("No tags were merged")
            continue
            
        for canonical, merged in sorted(clusters.items()):
            print(f"\n  {canonical} (canonical)")
            for tag in sorted(merged):
                print(f"    <- {tag}")

In [ ]:
display_clusters(image_results, "image")

In [ ]:
display_clusters(text_results, "text")

## Final Canonical Tags per Method

In [ ]:
def display_final_tags(results: dict, tag_type: str):
    """Display final canonical tags for each method."""
    print(f"\n{'='*70}")
    print(f"FINAL CANONICAL TAGS FOR {tag_type.upper()}")
    print(f"{'='*70}")
    
    for method, data in results.items():
        print(f"\n{'-'*50}")
        print(f"{method} ({len(data['canonical'])} tags)")
        print(f"{'-'*50}")
        for tag in sorted(data['canonical']):
            print(f"  {tag}")

In [ ]:
display_final_tags(image_results, "image")

In [ ]:
display_final_tags(text_results, "text")

## Threshold Sensitivity Analysis

See how different thresholds affect the number of final tags.

In [ ]:
def threshold_analysis(tags: list[str], tag_type: str):
    """Analyze how thresholds affect clustering."""
    thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
    
    results = []
    for thresh in thresholds:
        tfidf_count = len(set(standardize_tfidf(tags, thresh).values()))
        st_count = len(set(standardize_sentence_transformers(tags, thresh).values()))
        spacy_count = len(set(standardize_spacy(tags, thresh).values()))
        
        results.append({
            'Threshold': thresh,
            'TF-IDF': tfidf_count,
            'SentenceTransformers': st_count,
            'spaCy': spacy_count
        })
    
    df = pd.DataFrame(results)
    print(f"\nThreshold Analysis for {tag_type.upper()} tags (original: {len(tags)})")
    print(df.to_string(index=False))
    return df

In [ ]:
image_threshold_df = threshold_analysis(image_tags, "image")

In [ ]:
text_threshold_df = threshold_analysis(text_tags, "text")

## Recommendations

Based on the analysis:

1. **SentenceTransformers (MiniLM)** - Best for semantic similarity, captures meaning well
2. **TF-IDF** - Good for morphologically similar tags (same root words), zero external dependencies
3. **spaCy** - Middle ground, but requires downloading models

Adjust the `distance_threshold` parameter:
- Lower threshold (0.2-0.3) = more aggressive merging
- Higher threshold (0.5-0.7) = more conservative, fewer merges